In [0]:
spark.sql("USE CATALOG wsshubhamcontest")

DataFrame[]

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_supplier AS
SELECT DISTINCT
    l.LIFNR AS vendor_number,
    l.NAME1 AS vendor_name,
    l.LAND1 AS country,
    l.ORT01 AS city,
    l.PSTLZ AS postal_code,
    m.WAERS AS currency
FROM wsshubhamcontest.silver.lfa1 l
LEFT JOIN wsshubhamcontest.silver.lfm1 m ON l.LIFNR = m.LIFNR
""")
print("✅ dim_supplier created")

✅ dim_supplier created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_storage AS
SELECT DISTINCT
    WERKS AS plant,
    LGORT AS storage_location,
    LGOBE AS storage_description
FROM wsshubhamcontest.silver.t001l
""")
print("✅ dim_storage created")

✅ dim_storage created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_currency AS
SELECT DISTINCT
    FCURR AS from_currency,
    TCURR AS to_currency,
    KURST AS exchange_rate_type,
    GDATU AS valid_from_date
FROM wsshubhamcontest.silver.tcurf
""")
print("✅ dim_currency created")

✅ dim_currency created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_customer AS
SELECT DISTINCT
    Customer_ID AS customer_id,
    Customer_Description AS customer_description,
    Customer_Region AS customer_region,
    Customer_Country_Region AS customer_country,
    Market_Description AS market_description,
    Market_Region AS market_region,
    Market_Sub_Region AS market_sub_region
FROM wsshubhamcontest.silver.master_customer
""")
print("✅ dim_customer created")

✅ dim_customer created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_location AS
SELECT DISTINCT
    Location_ID AS location_id,
    Plant AS plant,
    Location_Description AS location_description
FROM wsshubhamcontest.silver.master_location
""")
print("✅ dim_location created")

✅ dim_location created


In [0]:
dim_location = spark.sql("""
    SELECT DISTINCT
        Location_ID,
        Plant,
        Location_Description,
        Plant_Description,
        Location_Region,
        Plant_Region,
        Location_Type,
        Plant_Type,
        Latitude,
        Longttitude
    FROM wsshubhamcontest.silver.master_location
""").dropDuplicates(["Plant"])

dim_location.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("wsshubhamcontest.gold.dim_location")

print("✅ dim_location fixed:", dim_location.count(), "rows")

✅ dim_location fixed: 9468 rows


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_inventory AS
SELECT
    d.MATNR AS material_number,
    d.WERKS AS plant,
    d.LGORT AS storage_location,
    d.LABST AS unrestricted_stock,
    b.CHARG AS batch,
    v.VERPR AS moving_avg_price,
    v.STPRS AS standard_price
FROM wsshubhamcontest.silver.mard d
LEFT JOIN wsshubhamcontest.silver.mchb b 
    ON d.MATNR = b.MATNR AND d.WERKS = b.WERKS AND d.LGORT = b.LGORT
LEFT JOIN wsshubhamcontest.silver.mbew v 
    ON d.MATNR = v.MATNR AND d.WERKS = v.BWKEY
""")
print("✅ fact_inventory updated")

✅ fact_inventory updated


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_demand_actual AS
SELECT
    d.id,
    d.period,
    d.period_id,
    d.material,
    d.market,
    d.actual_qty,
    d.actual_revenue,
    d.actual_orders,
    d.shipped_qty,
    d.delivered_qty,
    d.returns_qty,
    d.net_actual_qty,
    d.net_revenue,
    d.discounts,
    d.final_revenue,
    d.service_level
FROM wsshubhamcontest.silver.demand_actual d
""")
print("✅ fact_demand_actual created")

✅ fact_demand_actual created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_demand_forecast AS
SELECT
    id,
    period,
    period_id,
    material,
    market,
    consensus_demand,
    ibp_consensus_forecast,
    budget_volumes,
    sc_forecast_override,
    statistical_forecast,
    forecast_selector,
    upside,
    fcst_snapshot_1,
    fcst_snapshot_2,
    fcst_snapshot_3,
    fcst_snapshot_4,
    fcst_snapshot_5,
    fcst_snapshot_6,
    fcst_snapshot_7,
    fcst_snapshot_8,
    fcst_snapshot_9,
    fcst_snapshot_10,
    fcst_snapshot_11,
    fcst_snapshot_12,
    ibp_forecast,
    fcst_variance_snapshot_1,
    fcst_variance_snapshot_2,
    fcst_variance_snapshot_3,
    fcst_variance_perc_snapshot_1,
    fcst_variance_perc_snapshot_2,
    fcst_variance_perc_snapshot_3,
    ibp_consensus_forecast_prior
FROM wsshubhamcontest.silver.demand_forcast
""")
print("✅ fact_demand_forecast created")

✅ fact_demand_forecast created


In [0]:
display(spark.sql("SHOW TABLES IN wsshubhamcontest.gold"))

database,tableName,isTemporary
gold,dim_currency,false
gold,dim_customer,false
gold,dim_location,false
gold,dim_storage,false
gold,dim_supplier,false
gold,fact_demand_actual,false
gold,fact_demand_forecast,false
gold,fact_inventory,false
gold,fact_purchase_order,false


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_product AS
SELECT DISTINCT
    m.MATNR AS material_number,
    k.MAKTX AS material_description,
    m.MTART AS material_type,
    m.MATKL AS material_group,
    m.MEINS AS base_unit_of_measure,
    m.SPART AS division,
    m.PRDHA AS product_hierarchy
FROM wsshubhamcontest.silver.mara m
LEFT JOIN wsshubhamcontest.silver.makt k 
ON try_cast(m.MATNR AS STRING) = try_cast(k.MATNR AS STRING)
""")
print("✅ dim_product created")

✅ dim_product created


In [0]:
display(spark.sql("SHOW TABLES IN wsshubhamcontest.gold"))

database,tableName,isTemporary
gold,dim_currency,false
gold,dim_customer,false
gold,dim_location,false
gold,dim_product,false
gold,dim_storage,false
gold,dim_supplier,false
gold,fact_demand_actual,false
gold,fact_demand_forecast,false
gold,fact_inventory,false
gold,fact_purchase_order,false


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_customer AS
SELECT 
    Customer_ID AS customer_id,
    MAX(Customer_Description) AS customer_description,
    MAX(Customer_Region) AS customer_region,
    MAX(Customer_Country_Region) AS customer_country,
    MAX(Market_Description) AS market_description,
    MAX(Market_Region) AS market_region,
    MAX(Market_Sub_Region) AS market_sub_region
FROM wsshubhamcontest.silver.master_customer
GROUP BY Customer_ID
""")
print("✅ dim_customer fixed!")


spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_date AS
SELECT DISTINCT
    period AS period_id,
    CAST(SUBSTR(CAST(period AS STRING), 1, 4) AS INT) AS year,
    CAST(SUBSTR(CAST(period AS STRING), 5, 2) AS INT) AS month
FROM wsshubhamcontest.silver.demand_actual
""")
print("✅ dim_date created!")


spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_batch AS
SELECT DISTINCT
    m.MATNR AS material,
    m.CHARG AS batch,
    m.VFDAT AS expiration_date,
    m.LIFNR AS supplier,
    m.LICHA AS supplier_batch,
    m.LWEDT AS last_good_receipt,
    m.HSDAT AS manufacturing_date,
    m.QNDAT AS next_inspection_date,
    m.FVDT1 AS sell_by_date,
    m.ERSDA AS created_on,
    m.ERNAM AS created_by,
    m.LAEDA AS changed_on,
    m.AENAM AS changed_by
FROM wsshubhamcontest.silver.mch1 m
WHERE m.MATNR IS NOT NULL
""")
print("✅ dim_batch created!")



spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_customer_product AS
SELECT DISTINCT
    Customer_ID AS customer_id,
    Product_ID AS product_id,
    Market_Segment AS market_segment,
    Remaining_Shelf_Life AS remaining_shelf_life
FROM wsshubhamcontest.silver.master_customer_product
WHERE Customer_ID IS NOT NULL
""")
print("✅ dim_customer_product created!")





spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_location_product AS
SELECT DISTINCT
    Location_ID AS location_id,
    Product_ID AS product_id,
    Plant AS plant,
    Material_Description AS product_description,
    Plant_Exclusion_X AS plant_exclusion,
    Market_DC_X AS market_dc
FROM wsshubhamcontest.silver.master_location_product
WHERE Location_ID IS NOT NULL
""")
print("✅ dim_location_product created!")





spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_inventory_month_end_stock AS
SELECT
    h.MATNR AS material_number,
    h.WERKS AS plant,
    h.LGORT AS storage_location,
    h.CHARG AS batch,
    h.LFGJA AS fiscal_year,
    h.LFMON AS fiscal_month,
    h.CLABS AS unrestricted_qty,
    v.VERPR AS moving_avg_price,
    v.STPRS AS standard_price
FROM wsshubhamcontest.silver.mchbh h
LEFT JOIN wsshubhamcontest.silver.mbewh v 
    ON h.MATNR = v.MATNR AND h.WERKS = v.BWKEY
WHERE h.MATNR IS NOT NULL
""")
print("✅ fact_inventory_month_end_stock created!")

spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_inventory_monthly_snapshot AS
SELECT
    h.MATNR AS material_number,
    h.WERKS AS plant,
    h.LGORT AS storage_location,
    h.LFGJA AS fiscal_year,
    h.LFMON AS fiscal_month,
    h.LABST AS unrestricted_qty,
    v.VERPR AS moving_avg_price,
    v.STPRS AS standard_price
FROM wsshubhamcontest.silver.mardh h
LEFT JOIN wsshubhamcontest.silver.mbewh v 
    ON h.MATNR = v.MATNR AND h.WERKS = v.BWKEY
WHERE h.MATNR IS NOT NULL
""")
print("✅ fact_inventory_monthly_snapshot created!")




# # fact_batch_release_extern
# spark.sql("""
# CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_batch_release_extern AS
# SELECT
#     q.PRUEFLOS AS inspection_lot,
#     q.WERK AS plant,
#     q.MATNR AS material,
#     q.LIFNR AS supplier,
#     q.ART AS inspection_type,
#     q.BUDAT AS posting_date,
#     q.ENSTEHDAT AS creation_date,
#     q.LOSMENGE AS lot_quantity,
#     t.NAME1 AS plant_description
# FROM wsshubhamcontest.silver.qals q
# LEFT JOIN wsshubhamcontest.silver.t001w t ON q.WERK = t.WERKS
# WHERE q.PRUEFLOS IS NOT NULL
# """)
# print("✅ fact_batch_release_extern created!")

# fact_batch_release_extern
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_batch_release_extern AS
SELECT
    q.PRUEFLOS AS inspection_lot,
    q.WERK AS plant,
    q.MATNR AS material,
    q.LIFNR AS supplier,
    q.ART AS inspection_type,
    q.BUDAT AS posting_date,
    q.ENSTEHDAT AS creation_date,
    q.LOSMENGE AS lot_quantity,
    t.NAME1 AS plant_description
FROM wsshubhamcontest.silver.qals q
LEFT JOIN wsshubhamcontest.silver.t001w t ON q.WERK = t.WERKS
WHERE q.PRUEFLOS IS NOT NULL
""")
print("✅ fact_batch_release_extern done!")

# fact_batch_release_internal
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_batch_release_internal AS
SELECT
    q.PRUEFLOS AS inspection_lot,
    q.WERK AS plant,
    q.MATNR AS material,
    q.ART AS inspection_type,
    q.BUDAT AS posting_date,
    q.ENSTEHDAT AS creation_date,
    q.LOSMENGE AS lot_quantity,
    a.VBEWERTUNG AS usage_decision,
    t.NAME1 AS plant_description
FROM wsshubhamcontest.silver.qals q
LEFT JOIN wsshubhamcontest.silver.qave a ON q.PRUEFLOS = a.PRUEFLOS
LEFT JOIN wsshubhamcontest.silver.t001w t ON q.WERK = t.WERKS
WHERE q.PRUEFLOS IS NOT NULL
""")
print("✅ fact_batch_release_internal done!")



# fact_batch_release_internal
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_batch_release_internal AS
SELECT
    q.PRUEFLOS AS inspection_lot,
    q.WERK AS plant,
    q.MATNR AS material,
    q.ART AS inspection_type,
    q.BUDAT AS posting_date,
    q.ENSTEHDAT AS creation_date,
    q.LOSMENGE AS lot_quantity,
    a.VBEWERTUNG AS usage_decision,
    t.NAME1 AS plant_description
FROM wsshubhamcontest.silver.qals q
LEFT JOIN wsshubhamcontest.silver.qave a ON q.PRUEFLOS = a.PRUEFLOS
LEFT JOIN wsshubhamcontest.silver.t001w t ON q.WERK = t.WERKS
WHERE q.PRUEFLOS IS NOT NULL
""")
print("✅ fact_batch_release_internal created!")

# dim_uom
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.dim_uom AS
SELECT DISTINCT
    MATNR AS material_number,
    MEINS AS base_unit_of_measure,
    MTART AS material_type
FROM wsshubhamcontest.silver.mara
WHERE MEINS IS NOT NULL
""")
print("✅ dim_uom created!")

# Final check
display(spark.sql("SHOW TABLES IN wsshubhamcontest.gold"))




✅ dim_customer fixed!
✅ dim_date created!
✅ dim_batch created!
✅ dim_customer_product created!
✅ dim_location_product created!
✅ fact_inventory_month_end_stock created!
✅ fact_inventory_monthly_snapshot created!
✅ fact_batch_release_extern done!
✅ fact_batch_release_internal done!
✅ fact_batch_release_internal created!
✅ dim_uom created!


database,tableName,isTemporary
gold,dim_batch,false
gold,dim_currency,false
gold,dim_customer,false
gold,dim_customer_product,false
gold,dim_date,false
gold,dim_location,false
gold,dim_location_product,false
gold,dim_product,false
gold,dim_storage,false
gold,dim_supplier,false


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_purchase_order AS
SELECT
    k.EBELN AS po_number,
    p.EBELP AS po_item,
    k.LIFNR AS vendor_number,
    k.EKGRP AS purchasing_group,
    k.WAERS AS currency,
    k.BEDAT AS po_date,
    p.MATNR AS material_number,
    p.WERKS AS plant,
    p.MENGE AS quantity,
    p.NETPR AS net_price,
    p.NETWR AS net_value,
    e.EINDT AS delivery_date,
    e.MENGE AS scheduled_quantity
FROM wsshubhamcontest.silver.ekko k
LEFT JOIN wsshubhamcontest.silver.ekpo p ON k.EBELN = p.EBELN
LEFT JOIN wsshubhamcontest.silver.eket e ON p.EBELN = e.EBELN AND p.EBELP = e.EBELP
WHERE p.MATNR IS NOT NULL
""")
print("✅ fact_purchase_order fixed!")

✅ fact_purchase_order fixed!


In [0]:
%sql
SELECT COUNT(*), COUNT(vendor_number), COUNT(DISTINCT vendor_number)
FROM wsshubhamcontest.gold.fact_purchase_order;

SELECT COUNT(*), COUNT(vendor_number), COUNT(DISTINCT vendor_number)  
FROM wsshubhamcontest.gold.dim_supplier;

count(1),count(vendor_number),count(DISTINCT vendor_number)
10000,10000,10000


In [0]:
%sql
SELECT plant, COUNT(*) as cnt
FROM wsshubhamcontest.gold.dim_location
GROUP BY plant
HAVING COUNT(*) > 1;

plant,cnt
PL91671,2
PL37400,2
PL23612,2
PL90930,2
PL95548,2
PL72236,2
PL58380,2
PL81775,2
PL76755,2
PL43993,2


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_purchase_order AS
SELECT
    k.EBELN AS po_number,
    p.EBELP AS po_item,
    k.LIFNR AS vendor_number,
    k.EKGRP AS purchasing_group,
    k.WAERS AS currency,
    k.BEDAT AS po_date,
    p.MATNR AS material_number,
    p.WERKS AS plant,
    p.MENGE AS quantity,
    p.NETPR AS net_price,
    p.NETWR AS net_value,
    e.EINDT AS delivery_date,
    e.MENGE AS scheduled_quantity
FROM wsshubhamcontest.silver.ekko k
LEFT JOIN wsshubhamcontest.silver.ekpo p ON k.EBELN = p.EBELN
LEFT JOIN wsshubhamcontest.silver.eket e ON p.EBELN = e.EBELN AND p.EBELP = e.EBELP
""")
print("✅ done")

✅ done


In [0]:
%sql
SELECT plant, material_number, net_value 
FROM wsshubhamcontest.gold.fact_purchase_order 
LIMIT 5;

plant,material_number,net_value
null,null,null
null,null,null
null,null,null
null,null,null
null,null,null


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_purchase_order AS
SELECT
    p.EBELN AS po_number,
    p.EBELP AS po_item,
    p.WERKS AS plant,
    p.MATNR AS material_number,
    p.MENGE AS quantity,
    p.NETPR AS net_price,
    p.NETWR AS net_value,
    p.LGORT AS storage_location,
    p.MATKL AS material_group,
    p.MEINS AS unit_of_measure,
    p.BUKRS AS company_code,
    p.PSTYP AS item_category,
    p.MTART AS material_type
FROM wsshubhamcontest.silver.ekpo p
""")
print("✅ fact_purchase_order created")

✅ fact_purchase_order created


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE wsshubhamcontest.gold.fact_purchase_order AS
SELECT
    p.EBELN AS po_number,
    p.EBELP AS po_item,
    p.MATNR AS material_number,
    p.WERKS AS plant,
    p.MENGE AS quantity,
    p.NETPR AS net_price,
    p.NETWR AS net_value,
    p.BUKRS AS company_code,
    p.MEINS AS order_unit,
    e.EINDT AS delivery_date,
    e.MENGE AS scheduled_quantity,
    e.WEMNG AS delivered_quantity
FROM wsshubhamcontest.silver.ekpo p
LEFT JOIN wsshubhamcontest.silver.eket e ON p.EBELN = e.EBELN AND p.EBELP = e.EBELP
""")
print("✅ done")

✅ done


In [0]:
%sql
SELECT EBELN FROM wsshubhamcontest.silver.ekko LIMIT 3

EBELN
4595004919
4594876795
4578589106
